In [35]:
import numpy as np
import joblib

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error



In [36]:

pip install lightgbm

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [37]:
pip install catboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [38]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [39]:
import pandas as pd 
df = pd.read_csv('../data/crop_yield_data.csv')

In [45]:
import os
print(os.getcwd())

c:\Users\RAJESH\OneDrive\Desktop\AYP\notebooks


In [ ]:
print("Original Shape:", df.shape)
print("Columns:", df.columns.tolist())

Original Shape: (2596, 6)
Columns: ['Fertilizer', 'temp', 'N', 'P', 'K', 'yeild']


In [ ]:
df = df.rename(columns={'yeild': 'Yield'})  
df = df.dropna().drop_duplicates()

In [ ]:
X = df[['Fertilizer', 'temp', 'N', 'P', 'K']]
y = df['Yield']

In [ ]:
pip install --upgrade xgboost scikit-learn lightgbm catboost joblib

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.



   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 1.5 MB/s eta 0:01:10
    --------------------------------------- 1.6/101.7 MB 2.1 MB/s eta 0:00:49
    --------------------------------------- 2.4/101.7 MB 2.5 MB/s eta 0:00:41
   - -------------------------------------- 3.1/101.7 MB 2.7 MB/s eta 0:00:37
   - -------------------------------------- 4.2/101.7 MB 3.1 MB/s eta 0:00:32
   - -------------------------------------- 5.0/101.7 MB 3.2 MB/s eta 0:00:30
   -- ------------------------------------- 6.3/101.7 MB 3.5 MB/s eta 0:00:28
   -- ------------------------------------- 7.3/101.7 MB 3.7 MB/s eta 0:00:26
   --- ------------------------------------ 8.9/101.7 MB 4.1 MB/s eta 0:00:23
   ---- ----------------------------------- 10.2/101.7 MB 4.3 MB/s eta 0:00:22
   --

In [56]:
stack.fit(X, y)
joblib.dump(stack, 'crop_yield_model.pkl')  

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


['crop_yield_model.pkl']

In [57]:
# ===== TRAIN TEST SPLIT =====
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [58]:
# ===== SCALING =====
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [66]:
from pathlib import Path

save_dir = Path.cwd() / 'notebooks'
if not save_dir.exists():
    save_dir = (Path.cwd() / '..' / 'notebooks').resolve()
save_dir.mkdir(parents=True, exist_ok=True)

np.save(save_dir / 'X_train_scaled.npy', X_train_scaled)
np.save(save_dir / 'X_test_scaled.npy', X_test_scaled)
np.save(save_dir / 'y_train.npy', y_train)
np.save(save_dir / 'y_test.npy', y_test)
print(f"Saved NumPy arrays to: {save_dir}")


Saved NumPy arrays to: C:\Users\RAJESH\OneDrive\Desktop\AYP\notebooks


In [ ]:

# ================= BASE MODELS =================
rf = RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1)

xgb = XGBRegressor(
    tree_method='hist',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

lgb = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05
)

cat = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    verbose=0
)

In [ ]:
# ================= STACKING =================
models = [
    ('rf', rf),
    ('xgb', xgb),
    ('lgb', lgb),
    ('cat', cat)
]

stack = StackingRegressor(
    estimators=models,
    final_estimator=Ridge(),   # fast + stable
    n_jobs=-1,
    passthrough=True
)



In [ ]:
stack.fit(X, y)
joblib.dump(stack, '../models/crop_yield_model.pkl')
y_pred = stack.predict(X)
print("R² Score:", r2_score(y, y_pred))
print("MAE:", mean_absolute_error(y, y_pred))
print("MSE:", mean_squared_error(y, y_pred))    


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


R² Score: 0.9026106800592354
MAE: 0.4848460409550003
MSE: 0.365756744572309


In [ ]:
# ================= SAVE =================
joblib.dump(stack, '../models/best_model.pkl')
print("✅ Model Saved")

✅ Model Saved
